# OVCF End-to-End Training and New-Patient Prediction

这个 notebook 按两个阶段运行：

## 阶段 1：训练并保存机器学习模型

输入：`datasets/merged_patient_data.xlsx`，这是已经由大模型结构化后的患者级数据。

训练特征包含：

- 基本临床信息：`age`, `sex`, `height_cm`, `weight_kg`, `smoking_history`, `drinking_history`, `injury_history`。
- 图片/视频结构化特征：各体位照片和动作视频对应的 LLM `score`。

输出：`code_final/ovcf_rf_model_bundle.joblib`，里面包含训练好的模型、特征列、阈值、训练摘要、特征重要性和训练集参考统计。

## 阶段 2：新患者预测

输入：

- 手动填写新患者基本信息 `PATIENT_META`。
- 输入新患者图片/视频目录 `PATIENT_DIR`。

流程：

1. 自动识别目录中的图片/视频。
2. 调用 `code_final/prompt.py` 中的大模型 prompt 生成结构化 `score + explanation`。
3. 展开成与训练 Excel 同构的模型特征。
4. `joblib.load` 阶段 1 保存的模型包，输出 OVCF 概率、分类结果和解释表。

解释表会同时显示模型信号较强的临床变量和 LLM 变量。LLM 变量会附带结构化时的大模型理由；临床变量会标注为手动输入的基本信息。

注意：这是研究流程示例，不替代临床诊断。运行大模型调用前，请先设置 `DASHSCOPE_API_KEY` 环境变量。


In [1]:
# 依赖检查。若这里报 NumPy / pandas 二进制冲突，先在当前 kernel 中修复环境后再继续。
import importlib
import sys

REQUIRED = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "openpyxl": "openpyxl",
    "requests": "requests",
}

missing = []
for module_name, package_name in REQUIRED.items():
    try:
        importlib.import_module(module_name)
    except Exception as exc:
        missing.append((package_name, repr(exc)))

if missing:
    print("依赖导入失败：")
    for package_name, err in missing:
        print(f"- {package_name}: {err}")
    print("\n可尝试在 notebook 中运行：")
    print('%pip install --force-reinstall "numpy<2" pandas scipy scikit-learn openpyxl joblib requests')
    raise SystemExit("请先修复 Python 环境依赖。")

print(f"Python: {sys.version.split()[0]}")
print("依赖检查通过。")

Python: 3.11.14
依赖检查通过。


In [8]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "datasets").exists() and (PROJECT_ROOT.parent / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

CODE_DIR = PROJECT_ROOT / "code_final"
OUTPUT_ROOT = CODE_DIR / "patient_prediction_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "training_excel": PROJECT_ROOT / "datasets" / "merged_patient_data.xlsx",
    "prompt_py": CODE_DIR / "prompt.py",
    "model_bundle_path": CODE_DIR / "ovcf_rf_model_bundle.joblib",
    "output_root": OUTPUT_ROOT,
    "dashscope_model": "qwen-vl-max",
    "prediction_threshold_default": 0.50,
    "random_state": 42,
    "video_frame_count": 16,
    "request_timeout_sec": 600,
    "ffmpeg_bin": "ffmpeg",
    "transcode_videos": False,
    "auto_lookup_patient_meta": False,  # 新患者场景默认关闭；基本信息请手动填 PATIENT_META。
}

# 新患者入口：改这里即可。
# 这里是新患者基本信息，建议手动填写；它会和图片/视频 LLM 特征一起进入已训练模型。
# 若确实是在复现实验数据，可把 CONFIG['auto_lookup_patient_meta'] 改为 True，让 notebook 从训练 Excel 自动匹配。
PATIENT_DIR = PROJECT_ROOT / "datasets" / "positive1"
PATIENT_META = {
    # 新患者手动输入示例：
    "age": 60,
    "sex": "F",
    "height_cm": 158,
    "weight_kg": 80,
    "smoking_history": "无",
    "drinking_history": "无",
    "injury_history": "高能量外伤",
}

# 如果自动文件名识别不准，可手动指定。key 见 MEDIA_RULES，例如 {"lateral": "1.jpg", "supine_to_sit": "1.mp4"}
FILE_OVERRIDES = {}

# 已经存在结构化 JSON 时，可设为 False 跳过大模型调用，只做预测。
CALL_LLM = True
FORCE_REPROCESS = False

print("PROJECT_ROOT:", PROJECT_ROOT)
print("训练数据:", CONFIG["training_excel"])
print("患者目录:", PATIENT_DIR)
print("模型输出:", CONFIG["model_bundle_path"])

PROJECT_ROOT: D:\vscode_projects\ovcf
训练数据: D:\vscode_projects\ovcf\datasets\merged_patient_data.xlsx
患者目录: D:\vscode_projects\ovcf\datasets\positive1
模型输出: D:\vscode_projects\ovcf\code_final\ovcf_rf_model_bundle.joblib


In [3]:
import ast
import json
import re
from typing import Any, Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd

RANDOM_STATE = CONFIG["random_state"]

SCORE_COLUMNS = [
    "image_back__ParaspinalTensionAsymmetry__score",
    "image_back__PelvicRotation__score",
    "image_back__ScapularSymmetry__score",
    "image_back__SpinalVerticality__score",
    "image_frontal__CoronalCurvature__score",
    "image_frontal__GlobalBalance__score",
    "image_frontal__PelvicLeveling__score",
    "image_frontal__ShoulderSymmetry__score",
    "image_frontal__SpinalAlignment__score",
    "image_lateral__HeadPosition__score",
    "image_lateral__LumbarLordosis__score",
    "image_lateral__SagittalBalance__score",
    "image_lateral__ThoracicKyphosis__score",
    "image_lateral__TrunkInclination__score",
    "video_roll_left__HipShoulderCoordination__score",
    "video_roll_left__PainDuringMovement__score",
    "video_roll_left__RollingSmoothness__score",
    "video_roll_left__SupportHandUsage__score",
    "video_roll_right__HipShoulderCoordination__score",
    "video_roll_right__PainDuringMovement__score",
    "video_roll_right__RollingSmoothness__score",
    "video_roll_right__SupportHandUsage__score",
    "video_sit_to_supine__ArmAssistance__score",
    "video_sit_to_supine__ExecutionTime__score",
    "video_sit_to_supine__MotionCoordination__score",
    "video_sit_to_supine__PainResponseLevel__score",
    "video_sit_to_supine__TrunkControl__score",
    "video_supine_to_sit__ArmAssistance__score",
    "video_supine_to_sit__ExecutionTime__score",
    "video_supine_to_sit__MotionCoordination__score",
    "video_supine_to_sit__PainResponseLevel__score",
    "video_supine_to_sit__TrunkControl__score",
]

DEMO_COLUMNS = [
    "age",
    "sex",
    "height_cm",
    "weight_kg",
    "smoking_history",
    "drinking_history",
    "injury_history",
]

FEATURE_COLUMNS = DEMO_COLUMNS + SCORE_COLUMNS

ORDINAL_COLUMNS = {
    "video_roll_left__SupportHandUsage__score": "SupportHandUsage",
    "video_roll_right__SupportHandUsage__score": "SupportHandUsage",
    "video_sit_to_supine__ArmAssistance__score": "ArmAssistance",
    "video_supine_to_sit__ArmAssistance__score": "ArmAssistance",
}

ORDINAL_VALUE_MAPS = {
    "SupportHandUsage": {
        "无使用": 0.0,
        "无辅助": 0.0,
        "无需辅助": 0.0,
        "无需要辅助": 0.0,
        "单手辅助": 1.0,
        "单手支撑": 1.0,
        "双手用力推床": 2.0,
        "双手发力辅助": 2.0,
        "双手辅助": 2.0,
    },
    "ArmAssistance": {
        "无支撑": 0.0,
        "无辅助": 0.0,
        "无需辅助": 0.0,
        "部分支撑": 1.0,
        "部分辅助": 1.0,
        "明显依赖支撑": 2.0,
        "明显依赖": 2.0,
        "双手支撑": 2.0,
    },
}

NULL_STRINGS = {"", "nan", "none", "null", "信息缺失", "缺失", "na", "n/a", "-"}


def is_missing_value(x: Any) -> bool:
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    if isinstance(x, str) and x.strip().lower() in NULL_STRINGS:
        return True
    return False


def normalize_sex(x: Any) -> float:
    if is_missing_value(x):
        return np.nan
    s = str(x).strip().lower()
    mapping = {"m": 1, "male": 1, "男": 1, "1": 1, "f": 0, "female": 0, "女": 0, "0": 0}
    return float(mapping.get(s, np.nan))


def encode_binary_history(x: Any) -> float:
    if is_missing_value(x):
        return 0.0
    s = str(x).strip().lower()
    if s in {"无", "否", "no", "0", "false", "无明显", "无吸烟史", "无饮酒史"}:
        return 0.0
    if "无" in s and "有" not in s:
        return 0.0
    return 1.0


def encode_injury_history(x: Any) -> float:
    if is_missing_value(x):
        return 0.0
    s = str(x).strip().lower()
    if "高能量" in s or "车祸" in s:
        return 2.0
    if "低能量" in s or "平地" in s or "扭伤" in s:
        return 1.0
    if "无" in s or "no" in s or s == "0":
        return 0.0
    return 0.0


def encode_ordinal_value(x: Any, kind: str) -> float:
    if is_missing_value(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip()
    try:
        return float(s)
    except Exception:
        pass
    if s in ORDINAL_VALUE_MAPS[kind]:
        return ORDINAL_VALUE_MAPS[kind][s]
    for key, val in ORDINAL_VALUE_MAPS[kind].items():
        if key in s:
            return val
    return np.nan


def standardize_target(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    return s.map({"positive": 1, "ovcf": 1, "1": 1, "case": 1, "control": 0, "negative": 0, "healthy": 0, "0": 0})


def prepare_model_matrix(raw_df: pd.DataFrame, feature_columns: List[str]) -> pd.DataFrame:
    df = raw_df.copy()
    for col in feature_columns:
        if col not in df.columns:
            df[col] = np.nan

    df["sex"] = df["sex"].apply(normalize_sex)
    df["smoking_history"] = df["smoking_history"].apply(encode_binary_history)
    df["drinking_history"] = df["drinking_history"].apply(encode_binary_history)
    df["injury_history"] = df["injury_history"].apply(encode_injury_history)

    for col, kind in ORDINAL_COLUMNS.items():
        if col in df.columns:
            df[col] = df[col].apply(lambda v, k=kind: encode_ordinal_value(v, k))

    for col in feature_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df[feature_columns]


def load_training_dataframe(excel_path: Path) -> pd.DataFrame:
    df = pd.read_excel(excel_path)
    if "group" not in df.columns:
        raise KeyError("训练 Excel 必须包含 group 列。")
    df = df[~df["group"].astype(str).str.strip().str.lower().isin({"group", "nan", "none"})].copy()
    df["target"] = standardize_target(df["group"])
    df = df.dropna(subset=["target"]).copy()
    df["target"] = df["target"].astype(int)
    return df.reset_index(drop=True)


def pick_threshold_youden(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    from sklearn.metrics import roc_curve

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    idx = int(np.argmax(tpr - fpr))
    threshold = float(thresholds[idx])
    if not np.isfinite(threshold):
        threshold = CONFIG["prediction_threshold_default"]
    return threshold


def json_col_from_score_col(score_col: str) -> str:
    return score_col.replace("__score", "__json")


def normalize_lookup_text(x: Any) -> str:
    if is_missing_value(x):
        return ""
    s = str(x).strip().lower()
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"[()（）\[\]【】_\-—.,，。:：;；/\\]+", "", s)
    return s


def strip_modality_keywords(stem: str) -> str:
    s = str(stem)
    keywords = [
        "正位", "前位", "背位", "背侧", "后位", "侧位", "側位",
        "左翻", "右翻", "躺到坐", "卧到坐", "坐起来", "坐到躺", "坐到卧",
        "frontal", "front", "anterior", "back", "posterior", "lateral", "side",
        "roll_left", "left_roll", "roll_right", "right_roll",
        "supine_to_sit", "lie_to_sit", "sit_to_supine", "sit_to_lie",
    ]
    for key in keywords:
        s = re.sub(re.escape(key), "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"[_\-—.,，。:：;；/\\]+", "", s)
    return s.strip()


def collect_patient_lookup_candidates(patient_dir: Path) -> List[str]:
    patient_dir = Path(patient_dir)
    raw = [patient_dir.name, patient_dir.stem]
    if patient_dir.exists():
        for p in patient_dir.iterdir():
            if p.is_file():
                raw.extend([p.stem, strip_modality_keywords(p.stem)])
    candidates = []
    seen = set()
    for item in raw:
        norm = normalize_lookup_text(item)
        if norm and norm not in seen:
            seen.add(norm)
            candidates.append(norm)
    return candidates


def infer_patient_meta_from_training_excel(
    patient_dir: Path,
    training_excel: Path,
    manual_meta: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    manual_meta = manual_meta or {}
    if not CONFIG.get("auto_lookup_patient_meta", True):
        out = dict(manual_meta)
        if out:
            out["_meta_source"] = "manual_PATIENT_META"
        return out

    try:
        df = pd.read_excel(training_excel)
    except Exception as exc:
        print(f"[Warn] 无法读取训练 Excel 自动匹配基本信息: {exc}")
        return dict(manual_meta)

    if "group" in df.columns:
        df = df[~df["group"].astype(str).str.strip().str.lower().isin({"group", "nan", "none"})].copy()

    lookup_cols = [c for c in ["patient", "name", "patient_name", "姓名", "名字"] if c in df.columns]
    if not lookup_cols:
        return dict(manual_meta)

    candidates = collect_patient_lookup_candidates(patient_dir)
    if not candidates:
        return dict(manual_meta)

    matched_row = None
    matched_by = None
    for _, row in df.iterrows():
        for col in lookup_cols:
            value_norm = normalize_lookup_text(row.get(col, ""))
            if not value_norm:
                continue
            if value_norm in candidates or any(c in value_norm or value_norm in c for c in candidates if len(c) >= 2):
                matched_row = row
                matched_by = f"{col}={row.get(col)}"
                break
        if matched_row is not None:
            break

    inferred = {}
    if matched_row is not None:
        for col in DEMO_COLUMNS:
            if col in df.columns and not is_missing_value(matched_row.get(col)):
                inferred[col] = matched_row.get(col)
        inferred["_meta_source"] = f"training_excel:{matched_by}"
        print(f"[Info] 已自动匹配患者基本信息: {matched_by}")

    # 手动填写值覆盖自动匹配值。
    for key, value in manual_meta.items():
        if not is_missing_value(value):
            inferred[key] = value
    if manual_meta:
        inferred["_meta_manual_override"] = True
        inferred["_meta_source"] = inferred.get("_meta_source", "manual_PATIENT_META")
    return inferred



## 阶段 1：用 Excel 训练并保存机器学习模型

运行下面这个单元会读取 `CONFIG["training_excel"]`，训练模型，并保存到 `CONFIG["model_bundle_path"]`。后续新患者预测只加载这个 `.joblib` 模型包，不会重新训练。

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline


def classification_summary(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, Any]:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "auc": float(roc_auc_score(y_true, y_prob)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "sensitivity": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) else np.nan,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }


def build_reference_stats(X_numeric: pd.DataFrame, y: np.ndarray) -> Dict[str, Dict[str, float]]:
    stats = {}
    for col in X_numeric.columns:
        v = pd.to_numeric(X_numeric[col], errors="coerce")
        pos = v[y == 1]
        neg = v[y == 0]
        q75, q25 = np.nanpercentile(v, [75, 25]) if v.notna().any() else (np.nan, np.nan)
        iqr = q75 - q25
        stats[col] = {
            "median": float(np.nanmedian(v)) if v.notna().any() else np.nan,
            "iqr": float(iqr) if np.isfinite(iqr) and iqr > 0 else 1.0,
            "positive_mean": float(np.nanmean(pos)) if pos.notna().any() else np.nan,
            "control_mean": float(np.nanmean(neg)) if neg.notna().any() else np.nan,
        }
    return stats


def fit_and_save_model(excel_path: Path, model_bundle_path: Path, output_root: Path) -> Dict[str, Any]:
    train_df = load_training_dataframe(excel_path)
    X = prepare_model_matrix(train_df, FEATURE_COLUMNS)
    y = train_df["target"].to_numpy(dtype=int)

    model = Pipeline(
        [
            ("imputer", KNNImputer(n_neighbors=5)),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=600,
                    min_samples_leaf=2,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_prob = cross_val_predict(model, X, y, cv=cv, method="predict_proba", n_jobs=None)[:, 1]
    threshold_youden = pick_threshold_youden(y, cv_prob)
    cv_metrics_05 = classification_summary(y, cv_prob, 0.50)
    cv_metrics_youden = classification_summary(y, cv_prob, threshold_youden)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
    holdout_model = Pipeline(
        [
            ("imputer", KNNImputer(n_neighbors=5)),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=600,
                    min_samples_leaf=2,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    holdout_model.fit(X_train, y_train)
    holdout_prob = holdout_model.predict_proba(X_test)[:, 1]
    holdout_metrics = classification_summary(y_test, holdout_prob, threshold_youden)

    model.fit(X, y)
    estimator = model.named_steps["model"]
    importances = dict(zip(FEATURE_COLUMNS, estimator.feature_importances_.astype(float)))
    reference_stats = build_reference_stats(X, y)

    bundle = {
        "model": model,
        "feature_columns": FEATURE_COLUMNS,
        "score_columns": SCORE_COLUMNS,
        "demo_columns": DEMO_COLUMNS,
        "ordinal_columns": ORDINAL_COLUMNS,
        "ordinal_value_maps": ORDINAL_VALUE_MAPS,
        "threshold": float(threshold_youden),
        "feature_importances": importances,
        "reference_stats": reference_stats,
        "training_summary": {
            "n_samples": int(len(train_df)),
            "n_positive": int(y.sum()),
            "n_control": int(len(y) - y.sum()),
            "cv_metrics_at_0_50": cv_metrics_05,
            "cv_metrics_at_youden": cv_metrics_youden,
            "holdout_metrics_at_youden": holdout_metrics,
        },
    }

    model_bundle_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(bundle, model_bundle_path)

    output_root.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([cv_metrics_05, cv_metrics_youden, holdout_metrics], index=["CV_0.50", "CV_Youden", "Holdout_Youden"]).to_csv(
        output_root / "model_training_metrics.csv", encoding="utf-8-sig"
    )
    pd.DataFrame({"feature": list(importances), "importance": list(importances.values())}).sort_values(
        "importance", ascending=False
    ).to_csv(output_root / "model_feature_importance.csv", index=False, encoding="utf-8-sig")

    return bundle


bundle = fit_and_save_model(CONFIG["training_excel"], CONFIG["model_bundle_path"], CONFIG["output_root"])
print(f"[OK] 机器学习模型已训练并保存到: {CONFIG['model_bundle_path']}")
print(f"[OK] 训练指标已保存到: {CONFIG['output_root'] / 'model_training_metrics.csv'}")
bundle["training_summary"]


[OK] 机器学习模型已训练并保存到: D:\vscode_projects\ovcf\code_final\ovcf_rf_model_bundle.joblib
[OK] 训练指标已保存到: D:\vscode_projects\ovcf\code_final\patient_prediction_outputs\model_training_metrics.csv


{'n_samples': 206,
 'n_positive': 103,
 'n_control': 103,
 'cv_metrics_at_0_50': {'threshold': 0.5,
  'auc': 0.8259025355829956,
  'accuracy': 0.7475728155339806,
  'f1': 0.75,
  'sensitivity': 0.7572815533980582,
  'specificity': 0.7378640776699029,
  'tp': 78,
  'fp': 27,
  'tn': 76,
  'fn': 25},
 'cv_metrics_at_youden': {'threshold': 0.4514063610952404,
  'auc': 0.8259025355829956,
  'accuracy': 0.7766990291262136,
  'f1': 0.7927927927927928,
  'sensitivity': 0.8543689320388349,
  'specificity': 0.6990291262135923,
  'tp': 88,
  'fp': 31,
  'tn': 72,
  'fn': 15},
 'holdout_metrics_at_youden': {'threshold': 0.4514063610952404,
  'auc': 0.7959183673469388,
  'accuracy': 0.7142857142857143,
  'f1': 0.7692307692307693,
  'sensitivity': 0.9523809523809523,
  'specificity': 0.47619047619047616,
  'tp': 20,
  'fp': 11,
  'tn': 10,
  'fn': 1}}

In [10]:
import base64
import importlib.util
import mimetypes
import subprocess
import time

import requests

ACTION_TO_PREFIX = {
    "frontal": "image_frontal",
    "lateral": "image_lateral",
    "back": "image_back",
    "roll_left": "video_roll_left",
    "roll_right": "video_roll_right",
    "sit_to_supine": "video_sit_to_supine",
    "supine_to_sit": "video_supine_to_sit",
}

ACTION_TO_PROMPT_FILENAME = {
    "frontal": "image_frontal.txt",
    "lateral": "image_lateral.txt",
    "back": "image_back.txt",
    "supine_to_sit": "video_supine_to_sit.txt",
    "sit_to_supine": "video_sit_to_supine.txt",
    "roll_left": "video_roll.txt",
    "roll_right": "video_roll.txt",
}

MEDIA_RULES = {
    "frontal": {"ext": {".jpg", ".jpeg", ".png", ".bmp", ".webp"}, "keywords": ["frontal", "front", "anterior", "正位", "前位"]},
    "lateral": {"ext": {".jpg", ".jpeg", ".png", ".bmp", ".webp"}, "keywords": ["lateral", "side", "侧位", "側位"]},
    "back": {"ext": {".jpg", ".jpeg", ".png", ".bmp", ".webp"}, "keywords": ["back", "posterior", "背位", "背侧", "后位"]},
    "supine_to_sit": {"ext": {".mp4", ".mov", ".avi", ".mkv", ".m4v"}, "keywords": ["supine_to_sit", "lie_to_sit", "躺到坐", "卧到坐", "坐起来"]},
    "sit_to_supine": {"ext": {".mp4", ".mov", ".avi", ".mkv", ".m4v"}, "keywords": ["sit_to_supine", "sit_to_lie", "坐到躺", "坐到卧"]},
    "roll_left": {"ext": {".mp4", ".mov", ".avi", ".mkv", ".m4v"}, "keywords": ["roll_left", "left_roll", "左翻"]},
    "roll_right": {"ext": {".mp4", ".mov", ".avi", ".mkv", ".m4v"}, "keywords": ["roll_right", "right_roll", "右翻"]},
}


def load_prompt_content(prompt_py: Path) -> Dict[str, str]:
    spec = importlib.util.spec_from_file_location("ovcf_prompt_module", str(prompt_py))
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return dict(module.PROMPTS_CONTENT)


def find_patient_media(patient_dir: Path, overrides: Optional[Dict[str, str]] = None) -> Dict[str, Path]:
    patient_dir = Path(patient_dir)
    overrides = overrides or {}
    files = [p for p in patient_dir.iterdir() if p.is_file()]
    found: Dict[str, Path] = {}

    for action, rel in overrides.items():
        p = Path(rel)
        if not p.is_absolute():
            p = patient_dir / p
        if not p.exists():
            raise FileNotFoundError(f"FILE_OVERRIDES 指定的文件不存在: {action} -> {p}")
        found[action] = p

    for action, rule in MEDIA_RULES.items():
        if action in found:
            continue
        candidates = []
        for p in files:
            if p.suffix.lower() not in rule["ext"]:
                continue
            name = p.stem.lower()
            if any(str(k).lower() in name for k in rule["keywords"]):
                candidates.append(p)
        if candidates:
            candidates.sort(key=lambda p: len(p.name))
            found[action] = candidates[0]
    return found


def file_to_base64_data_url(file_path: Path) -> str:
    mime_type, _ = mimetypes.guess_type(str(file_path))
    if not mime_type:
        mime_type = "video/mp4" if file_path.suffix.lower() == ".mp4" else "image/jpeg"
    data = base64.b64encode(file_path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{data}"


def transcode_video(input_path: Path, output_dir: Path, ffmpeg_bin: str) -> Path:
    output_path = output_dir / f"{input_path.stem}_llm.mp4"
    cmd = [
        ffmpeg_bin,
        "-y",
        "-i",
        str(input_path),
        "-c:v",
        "libx264",
        "-crf",
        "25",
        "-preset",
        "medium",
        "-c:a",
        "aac",
        "-b:a",
        "96k",
        str(output_path),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return output_path


def extract_json_from_text(text: str) -> Dict[str, Any]:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.S)
        if match:
            return json.loads(match.group(0))
        raise


def parse_dashscope_response(resp: Dict[str, Any]) -> Dict[str, Any]:
    if "output" not in resp:
        return {"error": resp.get("error", "missing output"), "raw_response": resp}
    choices = resp.get("output", {}).get("choices", [])
    if not choices:
        return {"error": "empty choices", "raw_response": resp}
    content = choices[0].get("message", {}).get("content", [])
    text = None
    if isinstance(content, list):
        for part in content:
            if isinstance(part, dict) and "text" in part:
                text = part["text"]
                break
    elif isinstance(content, str):
        text = content
    if not text:
        return {"error": "no text content", "raw_response": resp}
    try:
        return extract_json_from_text(text)
    except Exception:
        return {"raw_content": text, "raw_response": resp}


def call_dashscope(media_path: Path, prompt_text: str, api_key: str, output_dir: Path) -> Dict[str, Any]:
    api_key = "sk-85fc4cc83aef43ebbf5bdc5459e5c64b"
    media_path = Path(media_path)
    is_video = media_path.suffix.lower() in MEDIA_RULES["supine_to_sit"]["ext"]
    media_for_upload = media_path
    if is_video and (CONFIG["transcode_videos"] or media_path.suffix.lower() != ".mp4"):
        media_for_upload = transcode_video(media_path, output_dir, CONFIG["ffmpeg_bin"])

    media_data_url = file_to_base64_data_url(media_for_upload)
    media_type = "video" if is_video else "image"
    payload = {
        "model": CONFIG["dashscope_model"],
        "input": {
            "messages": [{"role": "user", "content": [{"text": prompt_text}, {media_type: media_data_url}]}],
            "video_frame_sampling": {"strategy": "uniform", "sample_frame_count": CONFIG["video_frame_count"]},
        },
        "parameters": {"result_format": "message"},
    }
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    resp = requests.post(
        "https://dashscope.aliyuncs.com/api/v1/services/aigc/multimodal-generation/generation",
        headers=headers,
        json=payload,
        timeout=CONFIG["request_timeout_sec"],
    )
    resp.raise_for_status()
    return parse_dashscope_response(resp.json())


def get_or_create_structured_json(
    patient_dir: Path,
    patient_output_dir: Path,
    file_overrides: Optional[Dict[str, str]] = None,
    call_llm: bool = True,
    force_reprocess: bool = False,
) -> Tuple[Dict[str, Dict[str, Any]], Dict[str, Path]]:
    patient_output_dir.mkdir(parents=True, exist_ok=True)
    json_dir = patient_output_dir / "structured_json"
    json_dir.mkdir(parents=True, exist_ok=True)

    media = find_patient_media(patient_dir, file_overrides)
    prompts = load_prompt_content(CONFIG["prompt_py"])
    api_key = "sk-85fc4cc83aef43ebbf5bdc5459e5c64b"

    structured: Dict[str, Dict[str, Any]] = {}
    for action in ACTION_TO_PREFIX:
        out_json = json_dir / f"{action}.json"
        if out_json.exists() and not force_reprocess:
            structured[action] = json.loads(out_json.read_text(encoding="utf-8"))
            continue
        if action not in media:
            continue
        if not call_llm:
            continue
        if not api_key:
            raise RuntimeError("缺少 DASHSCOPE_API_KEY 环境变量，无法调用大模型。")

        prompt_text = prompts[ACTION_TO_PROMPT_FILENAME[action]].strip()
        print(f"结构化 {action}: {media[action].name}")
        last_error = None
        result = None
        for attempt in range(1, 4):
            try:
                result = call_dashscope(media[action], prompt_text, api_key, json_dir)
                break
            except Exception as exc:
                last_error = exc
                print(f"  attempt {attempt}/3 failed: {exc}")
                time.sleep(2)
        if result is None:
            result = {"error": str(last_error)}
        out_json.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
        structured[action] = result

    manifest = pd.DataFrame([{"action": action, "path": str(path), "used": action in structured} for action, path in media.items()])
    manifest.to_csv(patient_output_dir / "media_manifest.csv", index=False, encoding="utf-8-sig")
    return structured, media

In [6]:
def parse_indicator_payload(payload: Any) -> Dict[str, Any]:
    if isinstance(payload, dict):
        return payload
    if isinstance(payload, str):
        s = payload.strip()
        try:
            obj = json.loads(s)
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass
        return {"value": s}
    return {"value": payload}


def value_for_model(indicator: str, item: Dict[str, Any]) -> Any:
    # 训练集里 ArmAssistance / SupportHandUsage 是有序文本，因此预测时优先保留 value 再统一编码。
    if indicator in {"ArmAssistance", "SupportHandUsage"} and "value" in item:
        return item.get("value")
    if "score" in item:
        return item.get("score")
    if "value" in item:
        return item.get("value")
    return np.nan


def flatten_structured_json(
    structured: Dict[str, Dict[str, Any]],
    patient_id: str,
    patient_meta: Optional[Dict[str, Any]] = None,
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    row: Dict[str, Any] = {"patient": patient_id}
    if patient_meta:
        row.update(patient_meta)

    explanation_rows = []
    for action, data in structured.items():
        prefix = ACTION_TO_PREFIX.get(action)
        if not prefix or not isinstance(data, dict) or "error" in data:
            continue
        for indicator, raw_item in data.items():
            if indicator in {"error", "raw_content", "raw_response"}:
                continue
            item = parse_indicator_payload(raw_item)
            score_col = f"{prefix}__{indicator}__score"
            json_col = f"{prefix}__{indicator}__json"
            model_value = value_for_model(indicator, item)
            row[score_col] = model_value
            row[json_col] = json.dumps(item, ensure_ascii=False)
            explanation_rows.append(
                {
                    "patient": patient_id,
                    "action": action,
                    "prefix": prefix,
                    "indicator": indicator,
                    "score_column": score_col,
                    "value_for_model": model_value,
                    "score": item.get("score", np.nan),
                    "value": item.get("value", np.nan),
                    "explanation": item.get("explanation", ""),
                    "json": json.dumps(item, ensure_ascii=False),
                }
            )

    return row, pd.DataFrame(explanation_rows)


def build_patient_feature_files(
    patient_dir: Path,
    patient_meta: Optional[Dict[str, Any]],
    file_overrides: Optional[Dict[str, str]],
    call_llm: bool,
    force_reprocess: bool,
) -> Tuple[Dict[str, Any], pd.DataFrame, Path]:
    patient_dir = Path(patient_dir)
    patient_id = patient_dir.name
    patient_output_dir = CONFIG["output_root"] / patient_id
    structured, media = get_or_create_structured_json(
        patient_dir,
        patient_output_dir,
        file_overrides=file_overrides,
        call_llm=call_llm,
        force_reprocess=force_reprocess,
    )
    row, explanations = flatten_structured_json(structured, patient_id, patient_meta)
    feature_df = pd.DataFrame([row])
    feature_df.to_excel(patient_output_dir / "patient_structured_features.xlsx", index=False)
    feature_df.to_csv(patient_output_dir / "patient_structured_features.csv", index=False, encoding="utf-8-sig")
    explanations.to_csv(patient_output_dir / "patient_llm_explanations.csv", index=False, encoding="utf-8-sig")
    return row, explanations, patient_output_dir

In [7]:
def feature_display_name(score_col: str) -> str:
    if "__" not in score_col:
        return score_col
    prefix, indicator, _ = score_col.split("__", 2)
    return f"{prefix} / {indicator}"


def get_json_explanation(row: Dict[str, Any], score_col: str) -> Tuple[str, Any, Any]:
    json_col = json_col_from_score_col(score_col)
    raw = row.get(json_col, None)
    if is_missing_value(raw):
        return "", np.nan, np.nan
    item = parse_indicator_payload(raw)
    return str(item.get("explanation", "")), item.get("score", np.nan), item.get("value", np.nan)


def build_interpretability_table(row: Dict[str, Any], X_numeric: pd.DataFrame, bundle: Dict[str, Any], top_n: int = 8) -> pd.DataFrame:
    model = bundle["model"]
    feature_columns = bundle["feature_columns"]
    importances = bundle.get("feature_importances", {})
    ref = bundle.get("reference_stats", {})

    x_imputed = model.named_steps["imputer"].transform(X_numeric)[0]
    rows = []
    for i, feature in enumerate(feature_columns):
        raw_value = row.get(feature, np.nan)
        if is_missing_value(raw_value):
            continue
        stats = ref.get(feature, {})
        pos_mean = stats.get("positive_mean", np.nan)
        ctrl_mean = stats.get("control_mean", np.nan)
        median = stats.get("median", np.nan)
        iqr = stats.get("iqr", 1.0) or 1.0
        direction = np.sign(pos_mean - ctrl_mean) if np.isfinite(pos_mean) and np.isfinite(ctrl_mean) else 0.0
        z_like = (x_imputed[i] - median) / iqr if np.isfinite(median) else 0.0
        risk_signal = float(importances.get(feature, 0.0) * z_like * direction)
        if feature in SCORE_COLUMNS:
            explanation, llm_score, llm_value = get_json_explanation(row, feature)
            feature_type = "LLM_score"
        else:
            explanation, llm_score, llm_value = "临床基本信息特征，无 LLM explanation。", np.nan, raw_value
            feature_type = "clinical"
        rows.append(
            {
                "feature": feature,
                "feature_type": feature_type,
                "display": feature_display_name(feature),
                "raw_value": raw_value,
                "numeric_value_after_encoding": float(x_imputed[i]) if np.isfinite(x_imputed[i]) else np.nan,
                "model_importance": float(importances.get(feature, 0.0)),
                "positive_mean": pos_mean,
                "control_mean": ctrl_mean,
                "risk_signal": risk_signal,
                "llm_score": llm_score,
                "llm_value": llm_value,
                "llm_explanation": explanation,
            }
        )

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["abs_signal"] = out["risk_signal"].abs()
    out = out.sort_values(["risk_signal", "abs_signal"], ascending=[False, False]).reset_index(drop=True)
    return out.head(top_n)


def predict_patient_from_row(row: Dict[str, Any], bundle: Dict[str, Any], patient_output_dir: Path) -> Tuple[Dict[str, Any], pd.DataFrame]:
    X = prepare_model_matrix(pd.DataFrame([row]), bundle["feature_columns"])
    model = bundle["model"]
    probability = float(model.predict_proba(X)[0, 1])
    threshold = float(bundle.get("threshold", CONFIG["prediction_threshold_default"]))
    pred = int(probability >= threshold)

    interp = build_interpretability_table(row, X, bundle, top_n=10)
    missing_modalities = []
    for action, prefix in ACTION_TO_PREFIX.items():
        cols = [c for c in SCORE_COLUMNS if c.startswith(prefix + "__")]
        if cols and all(is_missing_value(row.get(c, np.nan)) for c in cols):
            missing_modalities.append(action)

    clinical_meta_used = {col: row.get(col, None) for col in DEMO_COLUMNS if not is_missing_value(row.get(col, None))}

    report = {
        "patient": row.get("patient", "unknown"),
        "clinical_meta_used": clinical_meta_used,
        "clinical_meta_source": row.get("_meta_source", "manual_or_not_found"),
        "ovcf_probability": probability,
        "threshold": threshold,
        "prediction": "OVCF" if pred == 1 else "control/non-OVCF",
        "missing_modalities": missing_modalities,
        "model_training_summary": bundle.get("training_summary", {}),
        "top_explanatory_features": interp.to_dict(orient="records") if not interp.empty else [],
        "note": "研究模型输出，不替代临床诊断；解释为模型特征贡献启发式排序，并结合 LLM 结构化理由。",
    }

    patient_output_dir.mkdir(parents=True, exist_ok=True)
    (patient_output_dir / "prediction_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    interp.to_csv(patient_output_dir / "prediction_explanation_table.csv", index=False, encoding="utf-8-sig")
    return report, interp


def run_end_to_end_patient_prediction(
    patient_dir: Path,
    patient_meta: Optional[Dict[str, Any]] = None,
    file_overrides: Optional[Dict[str, str]] = None,
    call_llm: bool = True,
    force_reprocess: bool = False,
    model_bundle_path: Optional[Path] = None,
) -> Tuple[Dict[str, Any], pd.DataFrame, pd.DataFrame]:
    model_bundle_path = model_bundle_path or CONFIG["model_bundle_path"]
    if not Path(model_bundle_path).exists():
        raise FileNotFoundError(f"模型文件不存在，请先运行训练单元: {model_bundle_path}")
    bundle = joblib.load(model_bundle_path)
    print(f"[OK] 已加载训练好的机器学习模型: {model_bundle_path}")
    resolved_patient_meta = infer_patient_meta_from_training_excel(
        patient_dir=patient_dir,
        training_excel=CONFIG["training_excel"],
        manual_meta=patient_meta or {},
    )
    missing_demo = [c for c in DEMO_COLUMNS if is_missing_value(resolved_patient_meta.get(c, None))]
    if missing_demo:
        print(f"[Warn] 新患者以下基本信息未填写，将由训练集插补器处理: {missing_demo}")
    row, llm_explanations, patient_output_dir = build_patient_feature_files(
        patient_dir=patient_dir,
        patient_meta=resolved_patient_meta,
        file_overrides=file_overrides or {},
        call_llm=call_llm,
        force_reprocess=force_reprocess,
    )
    report, explanation_table = predict_patient_from_row(row, bundle, patient_output_dir)
    return report, explanation_table, llm_explanations

## 阶段 2：新患者目录 + 手动基本信息 -> LLM 特征 -> 加载模型预测

在配置单元里手动填写 `PATIENT_META`，并把 `PATIENT_DIR` 指向新患者图片/视频目录。下面的预测函数会加载阶段 1 保存的模型包。

In [11]:
# 阶段 2：新患者预测入口。
# 先确认阶段 1 已经生成 CONFIG['model_bundle_path']。
# 改 PATIENT_DIR / PATIENT_META / FILE_OVERRIDES 后运行本单元。
# 如果没有 DASHSCOPE_API_KEY，但已经有缓存 JSON，可把 CALL_LLM=False。
report, explanation_table, llm_explanations = run_end_to_end_patient_prediction(
    patient_dir=PATIENT_DIR,
    patient_meta=PATIENT_META,
    file_overrides=FILE_OVERRIDES,
    call_llm=CALL_LLM,
    force_reprocess=FORCE_REPROCESS,
)

print(
    json.dumps(
        {
            "patient": report["patient"],
            "prediction": report["prediction"],
            "clinical_meta_used": report.get("clinical_meta_used", {}),
            "clinical_meta_source": report.get("clinical_meta_source", ""),
            "ovcf_probability": round(report["ovcf_probability"], 4),
            "threshold": round(report["threshold"], 4),
            "missing_modalities": report["missing_modalities"],
        },
        ensure_ascii=False,
        indent=2,
    )
)

explanation_table[
    [
        "display",
        "feature_type",
        "raw_value",
        "numeric_value_after_encoding",
        "model_importance",
        "risk_signal",
        "llm_explanation",
    ]
].head(10)

[OK] 已加载训练好的机器学习模型: D:\vscode_projects\ovcf\code_final\ovcf_rf_model_bundle.joblib
结构化 frontal: 毕明友 正位.jpg
结构化 back: 毕明友 背位.jpg
结构化 roll_left: 毕明友 左翻.mp4
结构化 roll_right: 毕明友 右翻.mp4
{
  "patient": "positive1",
  "prediction": "OVCF",
  "clinical_meta_used": {
    "age": 60,
    "sex": "F",
    "height_cm": 158,
    "weight_kg": 80,
    "smoking_history": "无",
    "drinking_history": "无",
    "injury_history": "高能量外伤"
  },
  "clinical_meta_source": "manual_PATIENT_META",
  "ovcf_probability": 0.7097,
  "threshold": 0.4514,
  "missing_modalities": [
    "lateral",
    "sit_to_supine",
    "supine_to_sit"
  ]
}


,display,feature_type,raw_value,numeric_value_after_encoding,model_importance,risk_signal,llm_explanation
0,injury_history,clinical,高能量外伤,2.00,0.110813,0.221626,临床基本信息特征，无 LLM explanation。
1,video_roll_left / SupportHandUsage,LLM_score,双手发力辅助,2.00,0.009945,0.009945,双手在翻身过程中持续推床面以协助完成动作，尤其在第二阶段明显发力，代偿作用显著
2,sex,clinical,F,0.00,0.007893,-0.000000,临床基本信息特征，无 LLM explanation。
3,smoking_history,clinical,无,0.00,0.009619,-0.000000,临床基本信息特征，无 LLM explanation。
4,drinking_history,clinical,无,0.00,0.004156,-0.000000,临床基本信息特征，无 LLM explanation。
5,video_roll_right / SupportHandUsage,LLM_score,单手辅助,1.00,0.011973,0.000000,右手轻触床面并施加小力推动，用于协助翻身启动，发力持续时间较短且强度较低
6,image_frontal / SpinalAlignment,LLM_score,0.42,0.42,0.000091,-0.000039,C7-S1连线向右侧偏移约1.8cm，躯干轴线出现轻中度侧移
7,video_roll_left / PainDuringMovement,LLM_score,0.53,0.53,0.022562,-0.000902,翻身时面部表情轻微紧张，动作节奏改变，中途有短暂停顿并伴随保护性收缩，提示中度疼痛干扰
8,video_roll_left / HipShoulderCoordination,LLM_score,0.58,0.58,0.013520,-0.005408,肩部启动早于髋部，相位差约20°，躯干呈现分段旋转，存在轻度分离运动
9,video_roll_left / RollingSmoothness,LLM_score,0.45,0.45,0.012671,-0.006335,翻身过程中存在明显减速，动作分为两个阶段完成，从仰卧到侧卧的过渡中出现短暂停顿，整体节奏不连贯


## 输出文件

运行完最后一个单元后，每个患者会在 `code_final/patient_prediction_outputs/<patient_dir_name>/` 下生成：

- `structured_json/*.json`：每个体位/动作的大模型原始结构化 JSON。
- `patient_structured_features.xlsx` / `.csv`：模型输入特征表，包含自动匹配或手动填写的年龄、性别、身高、体重、吸烟/饮酒/外伤史。
- `patient_llm_explanations.csv`：所有 LLM 指标的 score/value/explanation 明细。
- `prediction_report.json`：OVCF 概率、阈值、预测分类、缺失模态、解释性特征。
- `prediction_explanation_table.csv`：按当前患者模型信号排序后的解释表。

如果某些图片/视频文件名无法自动识别，请在配置单元中用 `FILE_OVERRIDES` 指定，例如：

```python
FILE_OVERRIDES = {
    "frontal": "毕明友 正位.jpg",
    "back": "毕明友 背位.jpg",
    "roll_left": "毕明友 左翻.mp4",
    "roll_right": "毕明友 右翻.mp4",
    "lateral": "1.jpg",
    "supine_to_sit": "1.mp4",
    "sit_to_supine": "2.mp4",
}
```